In [ ]:
# ===== installs =====
!pip -q install -U transformers datasets peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.4 MB/s eta 0:00:00


In [ ]:
# Upgrade torchao to a compatible version
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# ===== shared setup: model, tokenizer, data, helpers =====
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL  = "Qwen/Qwen2.5-0.5B"
print("device:", device)

tok = AutoTokenizer.from_pretrained(MODEL)

# code (Python), a real "domain"
ds = load_dataset("codeparrot/codeparrot-clean-valid", split="train", streaming=True)
key = "content"
texts = [ex[key] for i, ex in zip(range(500), ds)]
raw   = "\n\n".join(texts)

ids    = tok(raw, return_tensors="pt").input_ids[0]
BLOCK  = 256
blocks = torch.stack([ids[i:i+BLOCK] for i in range(0, len(ids) - BLOCK, BLOCK)])
print("training blocks:", tuple(blocks.shape))

def get_batch(bs=4):
    idx = torch.randint(0, blocks.size(0), (bs,))
    return blocks[idx].to(device)

def report_params(m, tag):
    tr  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m.parameters())
    print(f"[{tag}] trainable {tr:,} / {tot:,}  ({100*tr/tot:.2f}%)")

def train(model, tag, steps=60, lr=1e-5, bs=4):
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for step in range(steps):
        x   = get_batch(bs)
        out = model(input_ids=x, labels=x)   # next-token CE; HF shifts labels internally
        out.loss.backward()
        opt.step(); opt.zero_grad()
        if step % 20 == 0:
            print(f"[{tag}] step {step:3d} | loss {out.loss.item():.3f}")
    dt   = time.time() - t0
    peak = torch.cuda.max_memory_allocated()/1e9 if device == "cuda" else 0
    print(f"[{tag}] DONE  {dt:.1f}s | peak GPU mem {peak:.2f} GB")

device: cuda


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/401 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1221524 > 131072). Running this sequence through the model will result in indexing errors


training blocks: (4771, 256)


In [ ]:
def complete(model, prompt, max_new_tokens=80):
    model.eval()
    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tok.eos_token_id,
        )
    print(tok.decode(out[0], skip_special_tokens=True))
    print("-" * 60)

In [ ]:
prompts = [
    "def fibonacci(n):\n    ",
    "import numpy as np\n\ndef normalize(x):\n    ",
    "# Read a CSV file and return the number of rows\ndef count_rows(path):\n    ",
]

In [ ]:
baseline = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
for p in prompts:
    complete(baseline, p)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

def fibonacci(n):
     if n <= 1:
        return n
     else:
        return(fibonacci(n-1)+fibonacci(n-2))

n = int(input("Enter the number of fibonacci numbers you want: "))
print("The fibonacci sequence up to",n,"is:")
for i in range(n):
    print(fibonacci(i))
------------------------------------------------------------
import numpy as np

def normalize(x):
     """Normalize the data by subtracting the mean and dividing by the standard deviation.

     Args:
       x (numpy array): The data to normalize.

     Returns:
       numpy array: The normalized data.
     """
     mean = np.mean(x)
     std = np.std(x)
     return (x - mean) / std
------------------------------------------------------------
# Read a CSV file and return the number of rows
def count_rows(path):
     with open(path, 'r') as file:
         data = file.read().split('\n')
         return len(data)
------------------------------------------------------------


In [ ]:
# ===== RAW FULL CPT (updates every weight) =====
full = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
report_params(full, "FULL")            # ~100% trainable
train(full, "FULL")

# Optional: see gradient checkpointing in action — rerun this cell with these two lines
# uncommented BEFORE train(), and watch peak mem drop while time rises:
# full.gradient_checkpointing_enable(); full.config.use_cache = False

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[FULL] trainable 494,032,768 / 494,032,768  (100.00%)
[FULL] step   0 | loss 2.038
[FULL] step  20 | loss 1.252
[FULL] step  40 | loss 1.515
[FULL] DONE  92.6s | peak GPU mem 8.24 GB


In [ ]:
for p in prompts:
    complete(full, p)

def fibonacci(n):
     if n==1:
         return 0
     elif n==2:
         return 1
     else:
         return fibonacci(n-1)+fibonacci(n-2)
n=int(input("Enter number"))
print(fibonacci(n))
------------------------------------------------------------
import numpy as np

def normalize(x):
     x = x - np.mean(x)
     x = x / np.std(x)
     return x

def normalize_array(x):
     x = x - np.mean(x)
     x = x / np.std(x)
     return x

def normalize_matrix(x):
     x = x - np.mean(x)
     x = x / np.std(x)
     return x

def normalize_array2(x
------------------------------------------------------------
# Read a CSV file and return the number of rows
def count_rows(path):
     with open(path, 'r') as f:
        rows = f.read().splitlines()
        return len(rows)

def read_data(path, col):
    data = []
    with open(path, 'r') as f:
        for row in f:
            data.append([float(x) for x in row.split(',')])
    return data

def read_data_from_file(path):
    data =
---------------

In [ ]:
del full; torch.cuda.empty_cache()     # free memory before LoRA-CPT

In [ ]:
# ===== LoRA CPT (updates only tiny A/B adapters) =====
from peft import LoraConfig, get_peft_model

base = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
# cfg  = LoraConfig(
#     r=8, lora_alpha=16,
#     target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],   # attention + FFN
#     modules_to_save=["embed_tokens", "lm_head"],   # ← these train FULLY, no adapter
#     lora_dropout=0.05, task_type="CAUSAL_LM",
# )
cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, task_type="CAUSAL_LM",
)
lora = get_peft_model(base, cfg)
report_params(lora, "LoRA")            # well under 1% trainable
train(lora, "LoRA")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[LoRA] trainable 540,672 / 494,573,440  (0.11%)
[LoRA] step   0 | loss 1.703
[LoRA] step  20 | loss 1.296
[LoRA] step  40 | loss 1.911
[LoRA] DONE  67.2s | peak GPU mem 5.98 GB


In [ ]:
for p in prompts:
    complete(lora, p)

def fibonacci(n):
     if n<=1:
         return n
     else:
         return(fibonacci(n-1)+fibonacci(n-2))
n = int(input("enter the number"))
print(fibonacci(n))
------------------------------------------------------------
import numpy as np

def normalize(x):
     """
     Normalize a numpy array.
     """
     x = np.asarray(x)
     if x.ndim == 1:
          x = x.reshape((-1, 1))
     x -= x.mean()
     x /= x.std()
     return x
------------------------------------------------------------
# Read a CSV file and return the number of rows
def count_rows(path):
     with open(path, 'r') as file:
        count = 0
        for line in file:
            count += 1
        return count

# Read a CSV file and return the number of columns
def count_cols(path):
    with open(path, 'r') as file:
        count = 0
        for line in file:
            count += 1
        return
------------------------------------------------------------


In [ ]:
del lora; torch.cuda.empty_cache()     # free memory